# MCXcor Data Preparation
This notebook does preprocessing to build a wf_Seismogram collection that can serve as input for the MCXcor algorithm for final testing.

This notebook assumes source, channel, and site have been previously loaded into db using web services.  The version linked to this notebook is called "MCXcortesting_load_metadata".

In [1]:
import mspasspy.client as msc
mspass_client=msc.Client(database_name='MCXcorTesting')
db = mspass_client.get_database()

/N/u/pavlis/Quartz/.local/lib/python3.10/site-packages/distributed/node.py:177: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 38965 instead
  warnings.warn(


## Miniseed file indexing
We have files in two directories - each box here handles one of the different directories.

In [2]:
import os
import dask.bag as dbg
# remove the comment below if you need to restart this workflow 
# at this point c
#db.drop_collection('wf_miniseed')
# Note this dir value assumes the wf dir was created with 
# the previous command that also downloads the data from AWS
current_directory = os.getcwd()
dir = os.path.join(current_directory, 'wf')
dfilelist=[]
with os.scandir(dir) as entries:
    for entry in entries:
        if entry.is_file():
            dfilelist.append(entry.name)
print(dfilelist)
mydata = dbg.from_sequence(dfilelist)

mydata = mydata.map(db.index_mseed_file,dir=dir)
index_return = mydata.compute()

print("Size of index_return == number of waveforms handled=",len(index_return))

['Event_14.msd', 'Event_4.msd', 'Event_6.msd', 'Event_10.msd', 'Event_12.msd', 'Event_19.msd', 'Event_5.msd', 'Event_1.msd', 'Event_8.msd', 'Event_11.msd', 'Event_15.msd', 'Event_7.msd', 'Event_17.msd', 'Event_9.msd', 'Event_3.msd', 'Event_2.msd', 'Event_16.msd', 'Event_13.msd', 'Event_0.msd', 'Event_18.msd']
Size of index_return == number of waveforms handled= 20


In [3]:
import os
import dask.bag as dbg
# remove the comment below if you need to restart this workflow 
# at this point chttp://127.0.0.1:8888/lab?token=e4d539bd9e8fb93b9dc76554f5b30ab60c000c809593486d

#db.drop_collection('wf_miniseed')
# Note this dir value assumes the wf dir was created with 
# the previous command that also downloads the data from AWS
current_directory = os.getcwd()
dir = os.path.join(current_directory, 'wf_MCXcor')
dfilelist=[]
with os.scandir(dir) as entries:
    for entry in entries:
        if entry.is_file():
            dfilelist.append(entry.name)
print(dfilelist)
mydata = dbg.from_sequence(dfilelist)
mydata = mydata.map(db.index_mseed_file,dir=dir)
index_return = mydata.compute()
print("Size of index_return == number of waveforms handled=",len(index_return))

['Event_14.msd', 'Event_4.msd', 'Event_6.msd', 'Event_10.msd', 'Event_12.msd', 'Event_19.msd', 'Event_24.msd', 'Event_5.msd', 'Event_1.msd', 'Event_8.msd', 'Event_23.msd', 'Event_11.msd', 'Event_20.msd', 'Event_15.msd', 'Event_21.msd', 'Event_7.msd', 'Event_17.msd', 'Event_9.msd', 'Event_3.msd', 'Event_2.msd', 'Event_16.msd', 'Event_13.msd', 'Event_0.msd', 'Event_25.msd', 'Event_18.msd', 'Event_22.msd']
Size of index_return == number of waveforms handled= 26


## Normalization
We run normalize for miniseed indexing here.

In [4]:
from mspasspy.db.normalize import normalize_mseed
ret = normalize_mseed(db)
print("Number of documents processed in wf_miniseed=",ret[0])
print("Number of documents updated with channel cross reference id=",ret[1])
print("Number of documents updated with site cross reference id=",ret[2])

Number of documents processed in wf_miniseed= 59727
Number of documents updated with channel cross reference id= 59727
Number of documents updated with site cross reference id= 59727


## Define processing functions
The imports and functions below set up for processing loops below.

In [5]:
from mspasspy.algorithms.window import WindowData
from mspasspy.algorithms.signals import detrend

from mspasspy.algorithms.basic import ator
from mspasspy.ccore.algorithms.basic import TimeWindow
from mspasspy.ccore.utility import ErrorSeverity
from mspasspy.db.normalize import (normalize,
                                   ObjectIdMatcher,
                                   OriginTimeMatcher)
from mspasspy.algorithms.window import WindowData
from mspasspy.algorithms.resample import (ScipyResampler,
                                          ScipyDecimator,
                                          resample,
                                         )
from obspy.geodetics import gps2dist_azimuth,kilometers2degrees
from obspy.taup import TauPyModel
import time

def set_PStime(d,Ptimekey="Ptime",Stimekey="Stime",model=None):
    """
    Function to calculate P and S wave arrival time and set times 
    as the header (Metadata) fields defined by Ptimekey and Stimekey.
    Tries to handle some complexities of the travel time calculator                 
    returns when one or both P and S aren't calculatable.  That is 
    the norm in or at the edge of the core shadow.  
    
    :param d:  input TimeSeries datum.  Assumes datum's Metadata 
      contains stock source and channel attributes.  
    :param Ptimekey:  key used to define the header attribute that 
      will contain the computed P time.  Default "Ptime".
    :param model:  instance of obspy TauPyModel travel time engine. 
      Default is None.   That mode is slow as an new engine will be
      constructed on each call to the function.  Normal use should 
      pass an instance for greater efficiency.  
    """
    if d.live:
        if model is None:
            model = TauPyModel(model="iasp91") 
        # extract required source attributes
        srclat=d["source_lat"]
        srclon=d["source_lon"]
        srcz=d["source_depth"]
        srct=d["source_time"] 
        # extract required channel attributes
        stalat=d["channel_lat"]
        stalon=d["channel_lon"]
        staelev=d["channel_elev"]
        # set up and run travel time calculator
        georesult=gps2dist_azimuth(srclat,srclon,stalat,stalon)
        # obspy's function we just called returns distance in m in element 0 of a tuple
        # their travel time calculator it is degrees so we need this conversion
        dist=kilometers2degrees(georesult[0]/1000.0)
        arrivals=model.get_travel_times(source_depth_in_km=srcz,
                                            distance_in_degree=dist,
                                            phase_list=['P','S'])
        # always post this for as it is not cheap to compute
        # WARNING:  don't use common abbrevation delta - collides with data dt
        d['epicentral_distance']=dist
        # these are CSS3.0 shorthands s - station e - event
        esaz = georesult[1]
        seaz = georesult[2]
        # css3.0 names esax = event to source azimuth; seaz = source to event azimuth
        d['esaz']=esaz
        d['seaz']=seaz
        # get_travel_times returns an empty list if a P time cannot be 
        # calculated.  We trap that condition and kill the output 
        # with an error message
        if len(arrivals)==2:
            Ptime=srct+arrivals[0].time
            rayp = arrivals[0].ray_param
            Stime=srct+arrivals[1].time
            rayp_S = arrivals[1].ray_param
            d.put(Ptimekey,Ptime)
            d.put(Stimekey,Stime)
            # These keys are not passed as arguments but could be - a choice
            # Ray parameter is needed for free surface transformation operator
            # note tau p calculator in obspy returns p=R sin(theta)/V_0
            d.put("rayp_P",rayp)
            d.put("rayp_S",rayp_S)
        elif len(arrivals)==1:
            if arrivals[0].name == 'P':
                Ptime=srct+arrivals[0].time
                rayp = arrivals[0].ray_param
                d.put(Ptimekey,Ptime)
                d.put("rayp_P",rayp)
            else:
                # Not sure we can assume name is S
                if arrivals[0].name == 'S':
                    Stime=srct+arrivals[0].time
                    rayp_S = arrivals[0].ray_param
                    d.put(Stimekey,Stime)
                    d.put("rayp_S",rayp_S)
                else:
                    message = "Unexpected single phase name returned by taup calculator\n"
                    message += "Expected phase name S but got " + arrivals[0].name
                    d.elog.log_error("set_PStime",
                                     message,
                                     ErrorSeverity.Invalid)
                    d.kill()
                
    # Note python indents mean if an ensemble is marked dead this function just silenetly returns 
    # what it received doing nothing - correct mspass model
    return d
def cut_Pwindow(d,stime=-100.0,etime=500.0):
    """
    Window datum relative to P time window.  Time
    interval extracted is Ptime+stime to Ptime+etime.
    Uses ator,rtoa feature of MsPASS.
    """
    if d.live:
        if "Ptime" in d:
            ptime = d["Ptime"]
            d.ator(ptime)
            d = WindowData(d,stime,etime)
            d.rtoa()
    return d
   

## Define processing objects
This section does more imports and defines processing objects required in the main processing loop.

In [6]:
from obspy.taup import TauPyModel
from mspasspy.algorithms.signals import filter,detrend
from mspasspy.algorithms.resample import ScipyDecimator,ScipyResampler,resample
from mspasspy.db.normalize import OriginTimeMatcher,ObjectIdMatcher

ttmodel = TauPyModel(model="iasp91")

stime=-100.0
etime=500.0

decimator=ScipyDecimator(20.0)
resampler=ScipyResampler(20.0)

chan_matcher = ObjectIdMatcher(db)
source_matcher = OriginTimeMatcher(db,t0offset=300.0,tolerance=1000.0)

## Scalar processing
This loop does grungy stuff to cut up and set require metadata.  It is quite time consuming but written serial for this testing.  

In [7]:
t0 = time.time()
nlive=0
ndead=0
cursor=db.wf_miniseed.find({})
for doc in cursor:
    # the normalize matchers in this read were defined in the normalize section of this 
    # notebook.  Could cause problems if this box is run out of order
    d = db.read_data(doc,collection='wf_miniseed',normalize=[chan_matcher,source_matcher])
    d = detrend(d,type="constant")
    d = filter(d,'bandpass',freqmin=0.01,freqmax=2.0,zerophase=False)
    d = resample(d,decimator,resampler)
    # this function will run faster if passed an instance the TauP calculator (ttmodel)
    # If left as the default an instance is instantiated on each call to the function which is very inefficient.
    d = set_PStime(d,model=ttmodel)
    d = cut_Pwindow(d,stime,etime)
    if d.live:
        nlive += 1
    else:
        ndead += 1
        #error_log=d.elog.get_error_log()
        #print("Error message(s) posted to dead datum")
        #for e in error_log:
        #    print(e.message)
    db.save_data(d,collection='wf_TimeSeries',data_tag='serial_preprocessed')
t=time.time()    
print("Total scalar signal processing time=",t-t0)
print("Number of live data saved=",nlive)
print("number of data killed=",ndead)

/N/u/pavlis/Quartz/.local/lib/python3.10/site-packages/obspy/io/mseed/headers.py:805: InternalMSEEDWarning: readMSEEDBuffer(): Unexpected end of file when parsing record starting at offset 69632. The rest of the file will not be read.
  warnings.warn(_w, InternalMSEEDWarning)


Total scalar signal processing time= 3000.6308484077454
Number of live data saved= 43827
number of data killed= 15900


## Three component processing
Here we use the bundle module to create ensembles of Seismogram objects.  Then run the Seismogram objects through some preprocessing culminating in broadband_snr_QC.

In [8]:
import math
from mspasspy.ccore.seismic import SlownessVector
from mspasspy.algorithms.basic import free_surface_transformation
from mspasspy.ccore.utility import ErrorSeverity

def apply_FST(d,rayp_key="rayp_P",seaz_key='seaz',vp0=6.0,vs0=3.5):
    """
    Apply free surface transformation operator of Kennett (1983) to an input `Seismogram` 
    object.   Assumes ray parameter and azimuth data are stored as Metadata in thDefinee 
    input datum.  If the ray parameter or azimuth key are not defined an error 
    message will be posted and the datum will be killed before returning.  
    :param d:  datum to process
    :type d:  Seismogram
    :param rayp_key:   key to use to extract ray parameter to use to compute the 
    free surface transformation operator.  Note function assumes the ray parameter is
    spherical coordinate form:  R sin(theta)/V.   Default is "rayp_P".
    :param seaz_key:   key to use to extract station to event azimuth. Default is "seaz".
    :param vp0:  surface P wave velocity (km/s) to use for free surface transformation 
    
    :param vs0:  surface S wave velocity (km/s) to use for free surface transformation.
    """
    if d.is_defined(rayp_key) and d.is_defined(seaz_key):
        rayp = d[rayp_key]
        seaz = d[seaz_key]
        # Some basic seismology here.  rayp is the spherical earth ray parameter
        # R sin(theta)/v.  Free surface transformation needs apparent velocity 
        # at Earth's surface which is sin(theta)/v when R=Re.   Hence the following
        # simple convertion to get apparent slowness at surface  sin(theta)/v
        Re=6378.1
        umag = rayp/Re   # magnitude of slowness vector
        # seaz is back azimuth - slowness vector points in direction of propagation
        # with is 180 degrees away from back azimuth
        az = seaz + 180.0
        # component slowness vector components in local coordinates
        ux = umag * math.sin(az)
        uy = umag * math.cos(az)
        # FST implementation requires this special class called a Slowness Vector
        u = SlownessVector(ux,uy)
        d = free_surface_transformation(d,u,vp0,vs0)
    else:
        d.kill()
        message = "one of required attributes rayp_P or seaz were not defined for this datum"
        d.elog.log_error("apply_FST",message,ErrorSeverity.Invalid)
        
    return d

In [9]:
from mspasspy.algorithms.bundle import bundle_seed_data
from mspasspy.util.Undertaker import Undertaker
from mspasspy.algorithms.basic import rotate_to_standard
from mspasspy.algorithms.snr import broadband_snr_QC

noise_window=TimeWindow(-90.0,-5.0)

t0=time.time()
stedronsky=Undertaker(db)
srcids=db.wf_TimeSeries.distinct('source_id')
nsrc=len(srcids)
print("This run will process ",nsrc,
      " common source gathers into Seismogram objects")
for sid in srcids:
    query={'source_id' : sid,
           'data_tag' : 'serial_preprocessed'}
    nd=db.wf_TimeSeries.count_documents(query)
    print('working on ensemble with source_id=',sid)
    print('database contains ',nd,' documents == channels for this ensemble')
    cursor=db.wf_TimeSeries.find(query)
    # For this operation we only need channel metadata loaded by normalization
    # orientation data is critical (hang and vang attributes)
    ensemble = db.read_data(cursor,
                            normalize=[chan_matcher],
                           )
    #print('Number of TimeSeries objects constructed for this source=',len(ensemble.member))
    if ensemble.dead():
        print("This ensemble was marked dead by reader - skipping")
        continue
    ensemble=bundle_seed_data(ensemble)

    # The reader would do the following handling of dead data automatically
    # it is included here for demonstration purposes only
    # part of the lesson is handling of dead data
    #print('Number of (3C) Seismogram object created from input ensemble=',len(ensemble.member))
    [living,bodies]=stedronsky.bring_out_your_dead(ensemble)
    print('number of bundled Seismogram=',len(living.member))
    print('number of killed Seismogram=',len(bodies.member))
    # bury the dead if necessary
    if len(bodies.member)>0:
        stedronsky.bury(bodies)
    del bodies
    ensemble=db.save_data(ensemble,collection='wf_Seismogram',data_tag='serial_preprocessed',return_data=True)
    for d in ensemble.member:
        d = rotate_to_standard(d)
        d = apply_FST(d)
        d = broadband_snr_QC(d,
                      component=2,
                      use_measured_arrival_time=True,
                      measured_arrival_time_key="Ptime",
                      noise_window=noise_window,
                      kill_null_signals=True,
                     )
        # save with gridfs as processing time is not such an issue here
        d = db.save_data(d,collection='wf_Seismogram',data_tag="preprocessed")
t = time.time()
print("Time to do three component processing=",t-t0)

This run will process  42  common source gathers into Seismogram objects
working on ensemble with source_id= 673363152e6e2f5c8a107607
database contains  207  documents == channels for this ensemble
number of bundled Seismogram= 69
number of killed Seismogram= 0
working on ensemble with source_id= 673363162e6e2f5c8a107608
database contains  633  documents == channels for this ensemble
number of bundled Seismogram= 211
number of killed Seismogram= 0
working on ensemble with source_id= 673363162e6e2f5c8a107609
database contains  1287  documents == channels for this ensemble
number of bundled Seismogram= 429
number of killed Seismogram= 0
working on ensemble with source_id= 673363162e6e2f5c8a10760a
database contains  1242  documents == channels for this ensemble
number of bundled Seismogram= 414
number of killed Seismogram= 0
working on ensemble with source_id= 673363162e6e2f5c8a10760b
database contains  1131  documents == channels for this ensemble
number of bundled Seismogram= 377
number